# OncoSeg — Verify the `fix/review-findings` branch on Colab

This notebook runs the **post-fix** test + verification suite for OncoSeg on a Colab **GPU** runtime.
It needs **no dataset and no checkpoint** — it installs dependencies, runs the full pytest suite
(including the ~13 tests that get *skipped* locally because `monai`/`nibabel`/`pydicom`/`highdicom`
are missing), lints, and smoke-tests the core algorithm code paths I fixed on real `torch` tensors.

**Before running:** `Runtime → Change runtime type → Hardware accelerator → GPU`.

Run the cells top to bottom. Total time ≈ 3–5 min (mostly the `monai[all]` install).


## 0 · Check the runtime (GPU expected, not required)


In [ ]:
import sys, platform
print('Python:', sys.version.split()[0], '|', platform.platform())
try:
    import torch
    print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ModuleNotFoundError:
    print('torch not yet installed (Colab usually ships it). It will be installed in the next step.')

## 1 · Clone the fixed branch

Public repo, branch `fix/review-findings` (24 fix commits).


In [ ]:
import os
REPO = "https://github.com/danielchen26/OncoSeg-3D-Multi-Scale-Tumor-Segmentation-for-Automated-Treatment-Response-Assessment.git"
BRANCH = "fix/review-findings"
if not os.path.isdir('oncoseg'):
    !git clone --branch $BRANCH --depth 1 $REPO oncoseg
%cd oncoseg
!git log --oneline -1

## 2 · Install dependencies

`.[dev,serve,dicom]` pulls in `monai[all]`, `nibabel`, `fastapi`, `highdicom`/`pydicom`, `pytest`, `ruff`.
This is the exact extras CI uses (plus `dicom`), so the tests that skip on a bare machine will actually run here.

_(If pip prints a dependency-resolver warning about Colab's pre-installed packages, it's safe to ignore for tests.)_


In [ ]:
!pip -q install -e '.[dev,serve,dicom]'
print('\n--- import check ---')
for m in ['torch','monai','nibabel','fastapi','pydicom','highdicom','scipy','numpy']:
    try:
        __import__(m); print(f'  {m}: OK')
    except Exception as e:
        print(f'  {m}: MISSING ({e})')

## 3 · Run the full test suite

Expectation: **everything passes** and the ~13 tests that are *skipped* on a machine without
`monai`/`nibabel`/`pydicom`/`highdicom` now **run for real** (so the total `passed` count is higher than the 75
observed locally). `-rs` shows skip reasons (should be few/none here).


In [ ]:
!pytest tests/ -v --tb=short -rs

## 4 · Lint (ruff) — same check CI runs


In [ ]:
!ruff check src/ tests/ && echo 'ruff: clean'

## 5 · Smoke-test the fixed algorithm code paths (real tensors, no dataset)

This exercises the core fixes end-to-end on **random weights + synthetic volumes** — it confirms the code
*runs*, not that it is accurate (accuracy needs training). Covers:

- **F04** — MC-Dropout uncertainty runs on the trained *inline* `train_all.OncoSeg` (used to crash: `self.model.decoder`).
- **F16** — uncertainty is per-channel **binary** entropy, bounded by `ln 2 ≈ 0.693`.
- **F06** — RECIST longest diameter scans **all** slices (phantom: longest extent on a non-max-area slice).
- **F09** — `DeepSupervisionLoss` interpolates multi-scale predictions instead of crashing on shape mismatch.
- **F08** — best-checkpoint selection is **NaN-safe** (`nanmean` + `math.isnan` guard).


In [ ]:
import torch, numpy as np, math
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# ---- F04 + F16: MC-Dropout on the INLINE train_all architecture (the one that was actually trained) ----
from train_all import OncoSeg as InlineOncoSeg
from src.inference import Predictor
model = InlineOncoSeg(in_channels=4, num_classes=3, embed_dim=24, depths=(2,2,2,2),
                      num_heads=(3,6,12,24), deep_supervision=False).to(device).eval()
assert hasattr(model, 'decoders') and not hasattr(model, 'decoder'), 'inline model should have .decoders (plural)'
pred = Predictor(model=model, device=torch.device(device), roi_size=(64,64,64), mc_samples=4)
img = torch.rand(1,4,64,64,64, device=device)
unc = pred._estimate_uncertainty(img)            # F04: must NOT raise AttributeError
print('F04 MC-dropout ran, uncertainty shape:', unc.shape)
print('F16 entropy in [0, ln2]?  max=%.4f  (ln2=%.4f) ->' % (float(unc.max()), math.log(2)),
      'OK' if unc.max() <= math.log(2)+1e-3 else 'FAIL')

# ---- F06: RECIST longest diameter across all slices ----
from src.response.recist import RECISTMeasurer
m = RECISTMeasurer()
mask = np.zeros((64,64,8), np.uint8)
mask[10:40,10:40,0] = 1      # big-area slice, diagonal ~41mm
mask[30,5:55,1]    = 1       # small-area slice, true longest extent 49mm
d = m.longest_axial_diameter(mask, pixdim=(1.0,1.0,1.0))
print('F06 longest diameter = %.1f mm (expect ~49; pre-fix max-area-only gave ~41) ->' % d,
      'OK' if d > 45 else 'FAIL')

# ---- F09: deep-supervision loss interpolates multi-scale predictions ----
from src.training.losses import DeepSupervisionLoss, DiceCELoss
ds = DeepSupervisionLoss(DiceCELoss())
tgt = torch.zeros(1,3,32,32,32, device=device); tgt[:,0]=1
preds = [torch.randn(1,3,32,32,32, device=device),
         torch.randn(1,3,16,16,16, device=device),
         torch.randn(1,3,8,8,8,   device=device)]
loss = ds(preds, tgt)                            # F09: must NOT raise a shape error
print('F09 deep-supervision loss = %.4f  finite scalar? ->' % float(loss),
      'OK' if torch.isfinite(loss) and loss.dim()==0 else 'FAIL')

# ---- F08: NaN-safe best-checkpoint selection ----
def guarded_select(metric, best):  return (not math.isnan(metric)) and metric > best
row = np.array([0.71, 0.67, np.nan])             # an empty-ET subject -> NaN region
print('F08 plain mean is NaN? ', math.isnan(float(np.mean(row))),
      '| nanmean = %.4f' % float(np.nanmean(row)),
      '| saves with guard? ->', 'OK' if guarded_select(float(np.nanmean(row)), 0.0) else 'FAIL')

print('\nSmoke test complete.')

## 6 · (Optional) Re-derive the two CRITICAL statistics from the committed arrays

No model needed — this recomputes the numbers behind the doc fixes (F01 Wilcoxon, F02 foreground ECE,
F07 dominant failure region) directly from the committed `.npy` / JSON, so you can confirm the honest
numbers the README now reports.


In [ ]:
import numpy as np, json
from scipy.stats import wilcoxon
o = np.load('experiments/local_results/oncoseg_per_subject_dice.npy')  # cols [TC, WT, ET]
u = np.load('experiments/local_results/unet3d_per_subject_dice.npy')
print('per-subject arrays:', o.shape, '(val n =', o.shape[0], '-> split 388/96)')
for i,name in enumerate(['TC','WT','ET']):
    a,b = o[:,i], u[:,i]; mask = ~(np.isnan(a)|np.isnan(b)); a,b = a[mask], b[mask]
    p = wilcoxon(a, b, alternative='greater').pvalue
    print(f'  {name}: delta={ (a-b).mean():+.4f}  p(OncoSeg>UNet3D)={p:.4f}  OncoSeg wins {int((a>b).sum())}/{mask.sum()}')
om, um = np.nanmean(o,axis=1), np.nanmean(u,axis=1); mm = ~(np.isnan(om)|np.isnan(um))
print('  MEAN: p =', round(float(wilcoxon(om[mm],um[mm],alternative="greater").pvalue),4),
      '-> F01: NO region significant; WT favors UNet3D')

# F07: dominant failure region via nanmean
means = np.nanmean(o, axis=1); bottom = np.argsort(means)[:5]
opr, bpr = np.nanmean(o,axis=0), np.nanmean(o[bottom],axis=0)
rel = {n:(opr[i]-bpr[i])/opr[i] for i,n in enumerate(['TC','WT','ET'])}
print('  F07 relative drop (bottom-5):', {k:round(v,3) for k,v in rel.items()},
      '-> dominant =', max(rel, key=rel.get))

# F02: foreground vs pooled ECE from the committed calibration bins
d = json.load(open('experiments/local_results/uncertainty_metrics.json'))
print('  F02 pooled ECE =', d['ece_median_case'], '| foreground ECE =', d.get('ece_median_case_foreground'),
      '-> over-confident on tumor voxels')

## 7 · (Optional, slow) Train end-to-end for a couple of epochs

Uncomment to actually exercise the **training loop** on the real MSD Brain Tumour dataset. This downloads
**~7 GB** and trains a few epochs on the GPU (tens of minutes) — it verifies the seeded, NaN-guarded,
correctly-labelled training path runs end-to-end. It does **not** reproduce the paper's 50-epoch numbers.


In [ ]:
# # WARNING: downloads ~7GB and trains. Uncomment to run.
# !python train_local.py --epochs 2 --val-interval 1 --seed 42

---
**Interpreting results:** Section 3 is the headline — all tests should pass, with the locally-skipped
`monai`/`nibabel`/`pydicom`/`highdicom` tests now executing. Section 5 prints `OK` for each fixed code path.
Section 6 reproduces the honest statistics the docs now report. Paste the Section 3 + 5 output back and
I can confirm the fixes hold in a full CUDA + monai environment.
